### import libraries

In [2]:
import ee
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import datetime as dt

# Load environment variables from .env file
load_dotenv()

True

### initialize earth engine

In [3]:
ee.Authenticate()   # Authenticate the Earth Engine API

# Initialize the Earth Engine API
try:
    ee.Initialize(project=os.getenv("GEE_PROJECT"))
    print("Earth Engine initialized")
except Exception as e:
    print("Error initializing Earth Engine: ", e)
    exit(1)

Earth Engine initialized


### main function

In [8]:
# Define your locations
locations = [
    {'name': 'Jakarta', 'lon': 106.8456, 'lat': -6.2088},
    {'name': 'Bandung', 'lon': 107.6191, 'lat': -6.9175},
    {'name': 'Surabaya', 'lon': 112.7521, 'lat': -7.2575},
    # Add more locations here
]

# Date range
start_date = '2023-01-01'
end_date = '2023-12-31'

# Load ERA5-Land dataset
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')

# Select climate variables
climate_vars = era5.select([
    'total_precipitation_sum',                    # Rainfall (m)
    'temperature_2m',                              # Temperature (K)
    'dewpoint_temperature_2m',                    # Dewpoint temperature (K)
    'surface_solar_radiation_downwards_sum'       # Solar radiation (J/m²)
])

# Filter by date
filtered = climate_vars.filterDate(start_date, end_date)

# Function to extract data for one location
def extract_location_data(location):
    print(f"Processing {location['name']}...")
    
    # Create point geometry
    point = ee.Geometry.Point([location['lon'], location['lat']])
    
    # Function to extract values for each image
    def extract_values(image):
        # Reduce to point
        values = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=11132  # ERA5-Land native resolution (~11km)
        )
        
        # Get date
        date = ee.Date(image.get('system:time_start'))
        
        # Create feature with all properties
        return ee.Feature(None, {
            'date': date.format('YYYY-MM-dd'),
            'location': location['name'],
            'longitude': location['lon'],
            'latitude': location['lat'],
            'rainfall_m': values.get('total_precipitation_sum'),
            'temperature_k': values.get('temperature_2m'),
            'dewpoint_k': values.get('dewpoint_temperature_2m'),
            'solar_radiation_j_m2': values.get('surface_solar_radiation_downwards_sum')
        })
    
    # Map over all images
    features = filtered.map(extract_values)
    
    # Convert to list and get info
    feature_list = features.toList(features.size())
    return feature_list.getInfo()

# Collect data for all locations
all_data = []

for location in locations:
    location_data = extract_location_data(location)
    
    # Extract properties from each feature
    for feature in location_data:
        props = feature['properties']
        all_data.append(props)

# Convert to pandas DataFrame
df = pd.DataFrame(all_data)

# Convert units for better readability
df['rainfall_mm'] = df['rainfall_m'] * 1000  # Convert m to mm
df['temperature_celsius'] = df['temperature_k'] - 273.15  # Convert K to °C
df['dewpoint_celsius'] = df['dewpoint_k'] - 273.15  # Convert K to °C
df['solar_radiation_kw_m2'] = df['solar_radiation_j_m2'] / 3600000  # Convert J/m² to kWh/m² (daily sum)

# Calculate relative humidity from temperature and dewpoint
def calculate_relative_humidity(temp_c, dewpoint_c):
    # Using Magnus formula
    a = 17.27
    b = 237.7
    alpha_temp = (a * temp_c) / (b + temp_c)
    alpha_dewpoint = (a * dewpoint_c) / (b + dewpoint_c)
    rh = 100 * (np.exp(alpha_dewpoint) / np.exp(alpha_temp))
    return rh

df['relative_humidity_percent'] = calculate_relative_humidity(
    df['temperature_celsius'], 
    df['dewpoint_celsius']
)

# Select and reorder columns for final output
df_final = df[[
    'date', 'location', 'latitude', 'longitude',
    'rainfall_mm', 'temperature_celsius', 
    'relative_humidity_percent', 'solar_radiation_kw_m2'
]]

# Sort by location and date
df_final = df_final.sort_values(['location', 'date']).reset_index(drop=True)

# Save to CSV
output_filename = f'era5_climate_data_{start_date}_to_{end_date}.csv'
df_final.to_csv(output_filename, index=False)

print(f"\nData extraction complete!")
print(f"Total records: {len(df_final)}")
print(f"Saved to: {output_filename}")
print(f"\nFirst few rows:")
print(df_final.head(10))

# Display summary statistics
print(f"\nSummary statistics by location:")
print(df_final.groupby('location')[['rainfall_mm', 'temperature_celsius', 
                                      'relative_humidity_percent']].describe())

Processing Jakarta...
Processing Bandung...
Processing Surabaya...

Data extraction complete!
Total records: 1092
Saved to: era5_climate_data_2023-01-01_to_2023-12-31.csv

First few rows:
         date location  latitude  longitude  rainfall_mm  temperature_celsius  \
0  2023-01-01  Bandung   -6.9175   107.6191     2.423066            21.543223   
1  2023-01-02  Bandung   -6.9175   107.6191     1.882035            21.925013   
2  2023-01-03  Bandung   -6.9175   107.6191     7.984104            21.586671   
3  2023-01-04  Bandung   -6.9175   107.6191    12.897156            21.139139   
4  2023-01-05  Bandung   -6.9175   107.6191     5.073425            22.006347   
5  2023-01-06  Bandung   -6.9175   107.6191     0.925311            22.707457   
6  2023-01-07  Bandung   -6.9175   107.6191     7.611231            22.437875   
7  2023-01-08  Bandung   -6.9175   107.6191    28.938711            22.026498   
8  2023-01-09  Bandung   -6.9175   107.6191     7.018173            22.781253   
9 